# A Stale Allow Leaks Through a Shared Connector

The safety claim for policy mirroring is that a stale "allow" in Ranger is
harmless because the source platform still enforces underneath.

This notebook tests that claim on the deployed stack. It revokes a user at
Redshift while Ranger still holds the allow from its last sync, queries
through Trino, then runs the sync and queries again.

Trino's Redshift catalog logs in as one `connection-user`, so Redshift never
sees the end user. Expect the query to **succeed** after the revocation and
to be refused only once Ranger syncs. The same sequence runs as
`test_stale_allow_leaks_through_the_shared_connector` in
`tests/validation/test_scenario3_the_failure_modes.py`.


## Setup: Connect to Trino (governed) and Redshift (source)


In [ ]:
import os, logging, time
import trino, psycopg2, requests
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

load_dotenv()
logging.basicConfig(level=logging.INFO)

# Governed path: Trino → Redshift (Ranger enforced)
trino_conn = trino.dbapi.connect(
    host=os.getenv('TRINO_HOST'), port=int(os.getenv('TRINO_PORT', 8080)),
    user=os.getenv('TRINO_USER', 'test_user'), catalog='redshift', schema='federation'
)

# Direct path: psycopg2 → Redshift (platform-native only)
redshift_conn = psycopg2.connect(
    host=os.getenv('REDSHIFT_HOST'), port=int(os.getenv('REDSHIFT_PORT', 5439)),
    database=os.getenv('REDSHIFT_DATABASE'),
    user=os.getenv('REDSHIFT_USER'), password=os.getenv('REDSHIFT_PASSWORD'),
    sslmode='require'
)
display(Markdown('**Connected** to both Trino (governed) and Redshift (direct).'))

## Step 1: Verify current access works through the governed path


In [ ]:
# Step 1: Confirm governed access works through Trino → Redshift
cursor = trino_conn.cursor()
cursor.execute('SELECT count(*) FROM redshift.federation.ledger_entries')
rows = cursor.fetchone()
display(Markdown(f'**Governed path active:** {rows[0]:,} rows in `redshift.federation.ledger_entries` via Trino'))

# Also show a sample row to confirm columns are visible
cursor.execute('SELECT entry_id, token_id, amount, jurisdiction FROM redshift.federation.ledger_entries LIMIT 3')
sample = cursor.fetchall()
cols = [d[0] for d in cursor.description]
display(Markdown(f'**Sample columns:** {cols}'))
for row in sample:
    display(Markdown(f'- entry_id={row[0]}, token_id={row[1]}, amount={row[2]}, jurisdiction={row[3]}'))

## Step 2: Revoke access at the source (Redshift RBAC)

We revoke the grant directly at Redshift. Ranger still has the stale "allow" policy.
The Ranger sync hasn't run yet -- this simulates the sync gap.


In [ ]:
# Step 2: Grant at the source, sync Ranger, then revoke at the source
rs_cursor = redshift_conn.cursor()

demo_password = os.getenv('DEMO_ANALYST_PASSWORD')
if not demo_password:
    raise ValueError('DEMO_ANALYST_PASSWORD must be set in .env — refusing to use a hardcoded default')

# Create the demo_analyst user if not exists, grant access, then revoke
# Uses parameterized DDL via format-string with validated password (Redshift does not
# support parameterized CREATE USER). The password is validated to reject SQL metacharacters.
if any(c in demo_password for c in ("'", ";", "--", "/*")):
    raise ValueError("DEMO_ANALYST_PASSWORD contains disallowed characters (' ; -- /*)")

try:
    rs_cursor.execute("CREATE USER demo_analyst PASSWORD %s", (demo_password,))
    redshift_conn.commit()
except Exception:
    redshift_conn.rollback()  # user may already exist

rs_cursor.execute('GRANT USAGE ON SCHEMA federation TO demo_analyst')
rs_cursor.execute('GRANT SELECT ON federation.ledger_entries TO demo_analyst')
redshift_conn.commit()

import sys
sys.path.insert(0, '..')  # repository root, when run from notebooks/
from src.extractors.redshift_to_ranger import RedshiftExtractor
print(RedshiftExtractor().extract_and_push()['status'])
display(Markdown('**Step 2a:** `demo_analyst` has SELECT at Redshift, and Ranger has synced the allow. '
                 "Wait for Trino's Ranger plugin to load it before revoking."))
time.sleep(60)

# Now revoke — Ranger still has the stale "allow"
rs_cursor.execute('REVOKE SELECT ON federation.ledger_entries FROM demo_analyst')
redshift_conn.commit()
display(Markdown('**Step 2b:** SELECT revoked at Redshift. Ranger still holds the allow from its last sync.'))

## Step 3: Query through Trino as the revoked user

Ranger's stale policy allows `demo_analyst`. Redshift has revoked them, but
Trino connects to Redshift as the shared `connection-user`, so that
revocation is never checked.


In [ ]:
# Step 3: The revoked user queries through Trino before the Ranger sync
trino_analyst = trino.dbapi.connect(
    host=os.getenv('TRINO_HOST'), port=int(os.getenv('TRINO_PORT', 8080)),
    user='demo_analyst', catalog='redshift', schema='federation'
)

def federated_count():
    cursor = trino_analyst.cursor()
    cursor.execute('SELECT count(*) FROM redshift.federation.ledger_entries')
    return cursor.fetchone()[0]

try:
    rows = federated_count()
    display(Markdown(f'### Exposed: {rows:,} rows returned after the revocation'))
    display(Markdown('Redshift checked the connector account, not `demo_analyst`. '
                     'The data stays readable until Ranger syncs.'))
except Exception as e:
    display(Markdown(f'### Blocked\n```\n{str(e)[:500]}\n```'))
    display(Markdown('Blocked before the sync means the platform saw the end user '
                     '(identity passthrough) or Ranger already dropped the allow.'))


## Step 4: Sync Ranger and query again


In [ ]:
# Step 4: The sync deletes the revoked allow; the same query is refused
print(RedshiftExtractor().extract_and_push())
time.sleep(60)  # let Trino's Ranger plugin load the change
try:
    rows = federated_count()
    display(Markdown(f'**Still readable after the sync:** {rows:,} rows. Check the reconciliation output above.'))
except Exception as e:
    display(Markdown(f'**Refused after the sync:**\n```\n{str(e)[:300]}\n```'))
finally:
    trino_analyst.close()
    redshift_conn.close()
    trino_conn.close()


## What this shows

| Moment | Ranger | Redshift, for demo_analyst | Through Trino (shared connector) |
|---|---|---|---|
| Before revocation | Allow | Allow | Rows returned |
| After revocation, before sync | Allow (stale) | Revoked | **Rows returned** |
| After sync | No allow | Revoked | Refused |

With a shared connector account, the mirror is the only per-user control, so
its sync window is the exposure window. `src/governance/safety_model.py`
models the same result (`IdentityMode.SERVICE_ACCOUNT`), and
`src/governance/patterns.py` compares it with the alternatives.
